In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/300features_40minwords_10context
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/__results__.html
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/submission.csv
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/__notebook__.ipynb
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/__output__.json
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/custom.css
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier

In [3]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/300features_40minwords_10context
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/__results__.html
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/submission.csv
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/__notebook__.ipynb
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/__output__.json
/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/custom.css
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [4]:
from gensim.models import Word2Vec

model = Word2Vec.load(
    "/kaggle/input/notebooks/hexuanfcb/notebook869f5f8b86/300features_40minwords_10context"
)

In [5]:
print(model.wv.vectors.shape)

(16490, 300)


In [6]:
from sklearn.cluster import KMeans

word_vectors = model.wv.vectors

num_clusters = int(word_vectors.shape[0] / 5)

print("Number of words:", word_vectors.shape[0])
print("Number of clusters:", num_clusters)

Number of words: 16490
Number of clusters: 3298


In [7]:
print("Running K means")

kmeans_clustering = KMeans(
    n_clusters=num_clusters
)

idx = kmeans_clustering.fit_predict(word_vectors)

Running K means


In [8]:
print(len(idx))

16490


In [9]:
word_centroid_map = dict(
    zip(model.wv.index_to_key, idx)
)

print(len(word_centroid_map))

print("movie:", word_centroid_map["movie"])
print("good:", word_centroid_map["good"])
print("awful:", word_centroid_map["awful"])

16490
movie: 383
good: 764
awful: 2963


In [10]:
for cluster_id in [783, 1110, 2393]:
    words_in_cluster = [
        word
        for word, cid in word_centroid_map.items()
        if cid == cluster_id
    ]

    print("Cluster", cluster_id)
    print(words_in_cluster)
    print()

Cluster 783
['make']

Cluster 1110
['voice']

Cluster 2393
['million']



In [11]:
for cluster in range(0, 10):

    print("\nCluster %d" % cluster)

    words = []

    for word, cluster_id in word_centroid_map.items():
        if cluster_id == cluster:
            words.append(word)

    print(words)


Cluster 0
['spider']

Cluster 1
['stupidity', 'thrill', 'nostalgia', 'romp', 'silliness', 'fury', 'definition', 'rates', 'fright', 'hilarity', 'absurdity', 'spectacle', 'steer', 'fluff', 'filmmaking', 'excess', 'mediocrity', 'letdown', 'proportions', 'yarn', 'weirdness', 'awfulness', 'badness', 'idiocy', 'chock', 'tedium', 'ineptitude']

Cluster 2
['couldn']

Cluster 3
['believing', 'translated', 'transformed', 'sunk', 'tricked', 'lured']

Cluster 4
['shining', 'ish', 'resemble', 'pale', 'esquire', 'boogie', 'wee', 'boulevard']

Cluster 5
['lie', 'kidding', 'mistaken', 'assuming', 'tempted', 'referring', 'sucker', 'begging', 'suggesting', 'unsure', 'fanatic', 'believer', 'thankful', 'joking', 'dial', 'inclined', 'astonished', 'exaggerating', 'prude', 'trois']

Cluster 6
['nails']

Cluster 7
['overlooked', 'accepted', 'ignored', 'exposed', 'affected', 'explored', 'represented', 'defined', 'regarded', 'expressed', 'resolved', 'judged']

Cluster 8
['area', 'desert', 'river', 'jungle', 'm

In [12]:
def create_bag_of_centroids(wordlist, word_centroid_map):

    num_centroids = max(word_centroid_map.values()) + 1

    bag_of_centroids = np.zeros(
        num_centroids,
        dtype="float32"
    )

    for word in wordlist:
        if word in word_centroid_map:
            index = word_centroid_map[word]
            bag_of_centroids[index] += 1

    return bag_of_centroids

In [13]:
import pandas as pd

train = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

test = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

In [14]:
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "html.parser").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()

    if remove_stopwords:
        stops = set(stopwords.words("english"))
        words = [w for w in words if w not in stops]

    return words


def getCleanReviews(reviews):
    clean_reviews = []

    for review in reviews["review"]:
        clean_reviews.append(
            review_to_wordlist(review, remove_stopwords=True)
        )

    return clean_reviews


clean_train_reviews = getCleanReviews(train)
clean_test_reviews = getCleanReviews(test)

print(len(clean_train_reviews))
print(len(clean_test_reviews))

25000
25000


In [15]:
train_centroids = np.zeros(
    (train["review"].size, num_clusters),
    dtype="float32"
)

counter = 0

for review in clean_train_reviews:
    train_centroids[counter] = create_bag_of_centroids(
        review,
        word_centroid_map
    )
    counter += 1

test_centroids = np.zeros(
    (test["review"].size, num_clusters),
    dtype="float32"
)

counter = 0

for review in clean_test_reviews:
    test_centroids[counter] = create_bag_of_centroids(
        review,
        word_centroid_map
    )
    counter += 1

print(train_centroids.shape)
print(test_centroids.shape)

(25000, 3298)
(25000, 3298)


In [16]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100)

forest = forest.fit(
    train_centroids,
    train["sentiment"]
)

result = forest.predict(test_centroids)

In [17]:
print(result[:20])

[1 0 1 1 0 0 0 1 0 1 1 1 1 1 0 1 0 0 1 0]


In [18]:
output = pd.DataFrame(
    data={
        "id": test["id"],
        "sentiment": result
    }
)

output.to_csv(
    "/kaggle/working/submission.csv",
    index=False,
    quoting=3
)

In [19]:
print(output.shape)
print(output.head())

(25000, 2)
           id  sentiment
0  "12311_10"          1
1    "8348_2"          0
2    "5828_4"          1
3    "7186_2"          1
4   "12128_7"          0
